# 05 — IMU Hand-Crafted Features + Classical Machine Learning

We now have labeled fixed-length IMU windows. For the current example:

```text
X_windows shape: (217, 200, 6)
y_windows shape: (217,)

Walking       138
StairsDown     40
Standing       39
```

Each example is a **2-second, 6-channel IMU window**. This notebook follows the classical ML route:

```text
raw IMU window → hand-crafted features → feature table → scaling → classifier → evaluation
```

Later we will give the **same raw windows** to a 1D CNN and let the neural network learn its own features.

> **Important:** today's recording is from only one person. Any train/test split in this notebook is for checking the plumbing and learning the workflow. It is **not** a valid estimate of generalization to a new person.


## 0. Save the windowed dataset from Notebook 04

A CSV is excellent for ordinary 2D tables, but awkward for a 3D array such as `(windows, samples, channels)`.

For these windowed signals, use a compressed NumPy **`.npz`** file. At the end of Notebook 04, save:

```python
np.savez_compressed(
    "imu_windows.npz",
    X_windows=X_windows.astype(np.float32),
    y_windows=y_windows.astype(str),
    window_starts=window_starts.astype(np.float64),
    window_stops=window_stops.astype(np.float64),
    channel_names=np.array(CHANNELS),
    sample_rate_hz=np.array(TARGET_FS),
)
```

Why not the alternatives?

- **CSV:** great for a feature table, terrible for a 3D waveform array.
- **pickle:** convenient, but Python-specific and should not be loaded from untrusted sources.
- **HDF5:** powerful, but more machinery than we need here.
- **Parquet:** excellent for tabular metadata/features, but the raw 3D windows still need another representation.

For this course, `.npz` for waveform arrays and CSV/Parquet for ordinary tables is a clean combination.


## 1. Setup and load the windows

Put `imu_windows.npz` in the same directory as this notebook.


In [14]:
%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("imu_windows.npz")
assert DATA_PATH.exists(), (
    f"Could not find {DATA_PATH.resolve()}\n"
    "Export imu_windows.npz from Notebook 04 first."
)

data = np.load(DATA_PATH, allow_pickle=False)
X_windows = data["X_windows"]
y_windows = data["y_windows"].astype(str)
window_starts = data["window_starts"]
window_stops = data["window_stops"]
CHANNELS = data["channel_names"].astype(str).tolist()
FS = float(data["sample_rate_hz"])

print("X_windows:", X_windows.shape)
print("y_windows:", y_windows.shape)
print("Channels:", CHANNELS)
print("Sample rate:", FS, "Hz")
display(pd.Series(y_windows, name="activity").value_counts())


X_windows: (217, 200, 6)
y_windows: (217,)
Channels: ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
Sample rate: 100.0 Hz


activity
Walking       138
StairsDown     40
Standing       39
Name: count, dtype: int64

### Your turn

Before doing any machine learning, answer:

1. What does each dimension of `X_windows.shape` mean?
2. How long is each window in seconds?
3. Are the classes balanced?
4. Why might raw accuracy be a slightly misleading score here?


## 2. Look at representative raw windows

Before reducing each waveform to summary numbers, look at the thing we are about to summarize.


In [ ]:
def plot_window(index):
    t = np.arange(X_windows.shape[1]) / FS
    fig, ax = plt.subplots(figsize=(10, 4))
    for ch_i, ch_name in enumerate(CHANNELS):
        ax.plot(t, X_windows[index, :, ch_i], label=ch_name, alpha=0.8)
    ax.set_xlabel("Time within window (s)")
    ax.set_ylabel("Sensor value")
    ax.set_title(f"Window {index}: {y_windows[index]}")
    ax.legend(ncol=3)
    plt.show()

for activity in np.unique(y_windows):
    idx = np.where(y_windows == activity)[0][0]
    plot_window(idx)


### Your turn

Pick two windows from the **same** activity and plot them. Do they look identical? They should not.

That variability is exactly why we need features that capture useful structure without memorizing one particular waveform.


## 3. Start with simple time-domain features

For each of the six channels we will calculate mean, standard deviation, RMS, minimum, maximum, peak-to-peak range, and interquartile range.


In [ ]:
'''
Write a function called "basic_stats" that takes a variable x
and returns a dictionnary of stats including mean, std, rms, min,
max, range, and iqr, computed using np.
'''
#def basic...

# run it on the first window to check:
basic_stats(X_windows[0, :, 0])


### Your turn

For a walking accelerometer signal, what do you expect to happen to standard deviation, RMS, and peak-to-peak range compared with standing? Make a prediction before calculating anything.


## 4. Add acceleration and gyroscope magnitude

Individual XYZ axes depend on phone orientation. A vector magnitude collapses the three axes into one orientation-friendlier signal:

\[
|a| = \sqrt{a_x^2 + a_y^2 + a_z^2}
\]

Magnitude is **not** magic orientation-proofing, but it is less dependent on which phone axis happened to point forward.


In [ ]:
# Write a function vector_magnitude(window,indices)
# that takes in a window and a list of indices (eg ["ax", "ay", "az"]) 
# to operate on, and returns the sqrt of the sum of squares of those columns:
# COMPLETE:
#def vector_magnitude(window, indices):
#    return np.sqrt(...)

# now get out the gyro indices and accelerometer indices into lists
acc_indices = [CHANNELS.index(c) for c in ["ax", "ay", "az"]]
gyro_indices = [CHANNELS.index(c) for c in ["gx", "gy", "gz"]]

# now use vector magnitude on the first window as a test:
a_mag = vector_magnitude(X_windows[0], acc_indices)
g_mag = vector_magnitude(X_windows[0], gyro_indices)

# plot the magnitude signals
fig, ax = plt.subplots(figsize=(10, 4))
# create time vector the length of a_mag with spacing FS:
# COMPLETE:
# t = np.arange(...)
ax.plot(t, a_mag, label="|acceleration|")
ax.plot(t, g_mag, label="|gyroscope|")
ax.set_xlabel("Time within window (s)")
# why does y_windows[0] work?
ax.set_title(f"Magnitude signals: {y_windows[0]}")
ax.legend()
plt.show()


## 5. Add simple frequency-domain features

Walking and stairs are rhythmic behaviors. A Fourier transform lets us ask what frequency is strongest and how much power lies in a locomotor-frequency band.


In [ ]:
# Read this code, look up fft, but you can just use it. Let's focus on ML,
# not signal processing. But you should roughly know what fft and rfft are doing.
# Also some idea how the argmax thing works. Plot it!
# You should analyze one window outside the function to see what all the
# intermediate variables and functions are doing, plotting each step.
def spectral_features(x, fs, band=(0.5, 3.0)):
    x = x - np.mean(x)  # remove DC
    freqs = np.fft.rfftfreq(len(x), d=1/fs)
    power = np.abs(np.fft.rfft(x)) ** 2

    if len(power) > 1:
        dominant_index = np.argmax(power[1:]) + 1
        dominant_frequency = freqs[dominant_index]
    else:
        dominant_frequency = np.nan

    in_band = (freqs >= band[0]) & (freqs <= band[1])
    band_power = power[in_band].sum()
    total_power = power[1:].sum()
    relative_band_power = band_power / total_power if total_power > 0 else 0.0

    return {
        "dominant_frequency": dominant_frequency,
        "band_power_0p5_3": band_power,
        "relative_band_power_0p5_3": relative_band_power,
    }

spectral_features(a_mag, FS)


{'dominant_frequency': np.float64(0.5),
 'band_power_0p5_3': np.float32(71314.56),
 'relative_band_power_0p5_3': np.float32(0.8058525)}

### Your turn

Why did we subtract the mean before finding the dominant frequency? Try changing the frequency band from `0.5–3 Hz`. What biological behavior would you expect in the band you chose?


## 6. Build the complete hand-crafted feature table

Now we turn every `samples × channels` waveform window into **one row** of features.


In [ ]:
# Go ahead and use this function as it -> it's just data munging.
# Your previously written fucntinos above should have worked.
def extract_hand_features(X, channel_names, fs):
    rows = []
    acc_idx = [channel_names.index(c) for c in ["ax", "ay", "az"]]
    gyro_idx = [channel_names.index(c) for c in ["gx", "gy", "gz"]]

    for window in X:
        row = {}

        for ch_i, ch_name in enumerate(channel_names):
            stats = basic_stats(window[:, ch_i])
            for stat_name, value in stats.items():
                row[f"{ch_name}_{stat_name}"] = value

        signals = {
            "amag": vector_magnitude(window, acc_idx),
            "gmag": vector_magnitude(window, gyro_idx),
        }

        for signal_name, signal in signals.items():
            stats = basic_stats(signal)
            for stat_name, value in stats.items():
                row[f"{signal_name}_{stat_name}"] = value

            spec = spectral_features(signal, fs)
            for feature_name, value in spec.items():
                row[f"{signal_name}_{feature_name}"] = value

        rows.append(row)

    return pd.DataFrame(rows)

feature_df = extract_hand_features(X_windows, CHANNELS, FS)
feature_df["activity"] = y_windows
feature_df["window_start_s"] = window_starts
feature_df["window_stop_s"] = window_stops

print("Feature table shape:", feature_df.shape)
display(feature_df.head())


The dimensionality has collapsed from hundreds of raw waveform values per window to a much smaller collection of interpretable measurements. That is **feature engineering**.

### Your turn

Choose three feature columns and explain what physical property of motion each one is trying to capture. Then add **one feature of your own** to `extract_hand_features()`.


## 7. Explore features before fitting a classifier

Let's ask whether any individual features already separate activities.


In [ ]:
candidate_features = [
    "amag_std",
    "amag_range",
    "amag_dominant_frequency",
    "gmag_std",
    "gmag_dominant_frequency",
    # ADD YOUR NEW CUSTOM FEATURE HERE...
]

available = [f for f in candidate_features if f in feature_df.columns]

for feature in available:
    fig, ax = plt.subplots(figsize=(7, 4))
    activities = np.unique(y_windows)
    groups = [
        feature_df.loc[feature_df["activity"] == activity, feature]
        for activity in activities
    ]
    ax.boxplot(groups, tick_labels=activities)
    ax.set_ylabel(feature)
    ax.set_title(feature)
    plt.xticks(rotation=20)
    plt.show()


### Your turn

Which single feature appears most useful? Which two activities look hardest to separate with that feature alone?

Comment on the new feature you added. How does it's plot look? Will it be good for discriminating things?


## 8. PCA: can we see activity structure in feature space?

This is a little bit of a sanity check and a peek at dimensional reduction. Let's use PCA to see if activities cluster in and are separable.

PCA is **not** our classifier. We use it here to visualize the feature table. Because the features have different units and scales, we standardize them first.


In [ ]:
# Write code to make a list called "feature_columns"
# that contains only feature column names and not activity, window_start_s and window_stop_s
# do it with code not manually so that if you add/remove features you can re-run this
# notebook without it crashing or needing updates!
#feature_columns = [
#    ...
#    ...
#]

# get the feature columns in a numpy array called X_features using your list above and to_numpy()
# X_features = ...
# scale them using StandardScaler:
# X_scaled = ...
# run a PCA on them with n_components=2:
#X_pca = PCA(...

# Now let's plot!
fig, ax = plt.subplots(figsize=(8, 6))
# Run a for loop over the unique y_windows (activity names!)
#for activity in ...
#    # get a mask for the current activity so can color it:
#    mask = ...
#    plot those pca scores only with label=-the name, a bit of alpha:  
#    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=activity, alpha=0.7)

ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.set_title("PCA of hand-crafted IMU features")
ax.legend()
plt.show()


### Your turn

Do the classes form distinct clusters? If PCA does **not** show perfect separation, does that prove a classifier cannot work? Why not?


## 9. Put feature extraction inside a scikit-learn Pipeline

For modeling, wrap the calculation in an sklearn transformer:

```text
raw 200×6 window → IMUFeatureExtractor → StandardScaler → LogisticRegression
```


In [ ]:
# Oh crap. Need to teach OOP.
class IMUFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, channel_names=None, fs=100.0):
        self.channel_names = channel_names
        self.fs = fs

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return extract_hand_features(X, self.channel_names, self.fs).to_numpy()

    def get_feature_names_out(self, input_features=None):
        demo = extract_hand_features(X_windows[:1], self.channel_names, self.fs)
        return demo.columns.to_numpy()

logit_model = Pipeline([
    ("features", IMUFeatureExtractor(channel_names=CHANNELS, fs=FS)),
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

logit_model


## 10. A temporary one-person train/test split

🚨 **This is deliberately NOT our final validation strategy.**

With one subject, we cannot test generalization to an unseen person. Worse, our windows overlap, so neighboring windows can be near-duplicates. A random split can put highly related windows on both sides.

We are doing this **only to verify that the feature pipeline can learn anything at all**. When the class dataset is assembled, we will replace this with subject-wise splitting.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_windows,
    y_windows,
    test_size=0.25,
    random_state=42,
    stratify=y_windows,
)

logit_model.fit(X_train, y_train)
y_pred = logit_model.predict(X_test)

print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print()
print(classification_report(y_test, y_pred))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap=None)
plt.title("TEMPORARY one-subject confusion matrix")
plt.show()


### Your turn

Compare `class_weight="balanced"` with `class_weight=None`. Does the confusion matrix change? Why might balancing matter when Walking has many more windows than the other activities?


## 11. Inspect what Logistic Regression learned

Because we standardized the features first, coefficient magnitudes become reasonably comparable. Large coefficients show which features push predictions toward particular classes, but they are **not proof of biological causation**.


In [ ]:
feature_names = logit_model.named_steps["features"].get_feature_names_out()
classifier = logit_model.named_steps["classifier"]

for class_index, class_name in enumerate(classifier.classes_):
    coef = classifier.coef_[class_index]
    coef_table = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coef,
        "abs_coefficient": np.abs(coef),
    }).sort_values("abs_coefficient", ascending=False)

    print(f"\nTop features for: {class_name}")
    display(coef_table.head(10))


### Your turn

Pick one large coefficient. Can you tell a plausible physical story for why that feature might distinguish that activity? Then ask the skeptical question: could this actually be capturing phone placement, annotation quirks, or another nuisance variable?


## 12. Optional comparison: Random Forest

Logistic Regression is linear in feature space. A Random Forest can learn nonlinear thresholds and interactions between features, and it does not require feature scaling.


In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("features", IMUFeatureExtractor(channel_names=CHANNELS, fs=FS)),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
    )),
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Logistic Regression macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Random Forest macro F1:", f1_score(y_test, rf_pred, average="macro"))


Logistic Regression macro F1: 0.7984415584415584
Random Forest macro F1: 0.9020249629047744


# Final Your Turn

You now have the classical-ML path:

```text
raw waveform → hand-crafted statistics → frequency features → feature vector → Logistic Regression / Random Forest → activity prediction
```

Before the next notebook, write down:

1. Which hand-crafted features seemed most useful?
2. Which features are likely sensitive to phone orientation?
3. Which features might generalize well to a new person?
4. What information from the raw waveform did we throw away?
5. Why might a 1D CNN use information these summary features discard?

That last question is the bridge into deep learning.
